In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
vocab_size = 100     # Kitne unique tokens hain.
d_model = 64         # Embedding dimension.
num_heads = 4        # Multi-head attention me heads.
num_layers = 2       # Transformer blocks kitne stack honge.
seq_length = 10      # Maximum tokens per sequence.

In [4]:
embedding = nn.Embedding(vocab_size, d_model)
# 100 tokens
# each token → 64 dimension vector

In [25]:
# Example input
x = torch.randint(0, vocab_size, (1, seq_length))
print(x)

tensor([[63, 68, 75, 30, 58, 94, 63,  2, 46, 74]])


In [26]:
embedded = embedding(x)
print(embedded.shape)
# So sentence transform ho gaya: tokens → vectors

torch.Size([1, 10, 64])


In [41]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, seq_length):
        super().__init__()

        pe = torch.zeros(seq_length, d_model)  # matrix created:  positions × embedding dimension

        position = torch.arange(0, seq_length).unsqueeze(1)   # each token position

        div_term = torch.exp(                                     # This line basically creates frequency values.
            torch.arange(0, d_model, 2) * (-torch.log(torch.tensor(10000.0)) / d_model)  # sin(position * frequency),  cos(position * frequency)
        )                                                                                # short distance relations, long distance relations
        
        pe[:, 0::2] = torch.sin(position * div_term)   # even dimensions → sin
                                                                              # This creates unique position signatures.
        pe[:, 1::2] = torch.cos(position * div_term)  # odd dimensions → cos

        pe = pe.unsqueeze(0)     # [1, seq_length, d_model]

        self.register_buffer("pe", pe)   # store positional encoding in model but do not train it    It becomes fixed positional encoding.

    def forward(self, x):

        x = x + self.pe[:, :x.size(1)]    # token_embedding + position_embedding

        return x

# token_embedding + positional_encoding

In [42]:
pos_encoder = PositionalEncoding(d_model, seq_length)
embedded = embedding(x)
embedded = pos_encoder(embedded)

print(embedded.shape)

torch.Size([1, 10, 64])


In [50]:
import math
class SelfAttention(nn.Module):

    def __init__(self, d_model):
        super().__init__()

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)

    def forward(self, x):

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        return Q, K, V

        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / math.sqrt(Q.size(-1))
        weights = torch.softmax(scores, dim=-1)
        output = torch.matmul(weights, V)

        return output

In [51]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, num_heads):
        super().__init__()

        self.num_heads = num_heads
        self.d_model = d_model

        self.head_dim = d_model // num_heads

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)

        self.fc_out = nn.Linear(d_model, d_model)

In [60]:
def forward(self, x):

    batch_size = x.shape[0]

    Q = self.Wq(x)       # Compute Q K V
    K = self.Wk(x)       # [batch, seq, d_model]
    V = self.Wv(x)       # [2,10,64]

    Q = Q.view(batch_size, -1, self.num_heads, self.head_dim)     # Split Into Heads
    K = K.view(batch_size, -1, self.num_heads, self.head_dim)     # [batch, seq, heads, head_dim]
    V = V.view(batch_size, -1, self.num_heads, self.head_dim)     # [2,10,8,8]

    Q = Q.transpose(1,2)     # Rearrange Dimensions
    K = K.transpose(1,2)     # [batch, heads, seq, head_dim]
    V = V.transpose(1,2)     # [2,8,10,8]

    scores = torch.matmul(Q, K.transpose(-2, -1))   # attention computation
    scores = scores / math.sqrt(self.head_dim)      
    weights = torch.softmax(scores, dim=-1)
    out = torch.matmul(weights, V)

    out = out.transpose(1,2)   # combine heads   [batch, seq, heads, head_dim]    [2,10,8,8]
    out = out.contiguous().view(batch_size, -1, self.d_model)  # Now flatten   [2,10,64]

    out = self.fc_out(out)    # Because concatenated heads need mixing

    return out

In [61]:
class TransformerBlock(nn.Module):

    def __init__(self, d_model, heads, forward_expansion):
        super().__init__()

        self.attention = MultiHeadAttention(d_model, heads)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, forward_expansion * d_model),
            nn.ReLU(),
            nn.Linear(forward_expansion * d_model, d_model)
        )

    def forward(self, x):

        attention = self.attention(x)

        x = self.norm1(attention + x)

        forward = self.feed_forward(x)

        out = self.norm2(forward + x)

        return out

In [62]:
class MiniTransformer(nn.Module):

    def __init__(self, vocab_size, d_model, heads, num_layers, seq_length):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)

        self.positional = PositionalEncoding(d_model, seq_length)

        self.layers = nn.ModuleList(
            [TransformerBlock(d_model, heads, 4) for _ in range(num_layers)]
        )

        self.fc_out = nn.Linear(d_model, vocab_size)


    def forward(self, x):

        x = self.embedding(x)

        x = self.positional(x)

        for layer in self.layers:
            x = layer(x)

        out = self.fc_out(x)

        return out